# 🧹 Build Cleaning Rules (Resumes + JDs)

## Goals
In this notebook, we will:

✔ Inspect noisy text patterns  
✔ Create modular cleaning functions  
✔ Normalize common formatting issues  
✔ Prepare text for NLP processing  

At the end, we will output a reusable `clean_text()` function.

---

## 🔗 Load previous extracted text

We will re-use the extraction utilities from Notebook 01.

In [1]:
import re
import unicodedata
from pathlib import Path

DATASET_RESUME_DIR = Path("../../dataset/raw/dataset_noisy_pdfs/resumes_pdf")
DATASET_JD_DIR = Path("../../dataset/raw/dataset_noisy_pdfs/jds_pdf")

# reuse extraction function
def extract_pdf_text(pdf_path):
    from pdfminer.high_level import extract_text
    return extract_text(pdf_path) or ""

# 1️⃣ Inspect raw text from multiple docs

We sample 5 resumes + 5 JDs.

In [2]:
sample_files = list(DATASET_RESUME_DIR.glob("*.pdf"))[:5] + list(DATASET_JD_DIR.glob("*.pdf"))[:5]

docs = {f.name: extract_pdf_text(f) for f in sample_files}
docs

{'resume_001.pdf': 'ALLISON HILL — Environmental health practitioner\nPhone: +91-8410529190   Email: jillrhodes@example.net\nSUMMARY:\nInstead ahead despite measure ago curret practice nation.\nPROJECTS:\n* Optimized model using Leadership, reducing memory usage by 15%. H\nSKILLS:\nProject Management, Angular, Attention to Detail, Leadership, Node.js, React\nEXPERIENCE:\nEnglish as a second language teache\nr at Lawrence-Pacheco (2019 - 2022)\nﬁ Designed using Project Management to improve accuracy.\n- Enhanced using Leadership to improve memory usage.\nAccounting technician at Wolfe LLC (2014 - 2016)\nn Integrated using Angular to improve accuracy.\n- Integrated using Leadership to improev throughput.\nﬁ Debbugged using React to improve processing speed.\nHealth physicist at Lewis LLC (2019 - 2024)\n- Optimized using Angular to improve memory usage.\n- Implemented using Angular to improve throughput.\n- Tested using Project Management to improve processing speed.\nEDUCATION:\nB.S. in 

# 2️⃣ Universal text cleaning rules (modular!)

We will progressively add rules:
- strip extra whitespace
- normalize unicode
- remove weird symbols
- unify newlines

In [3]:
def basic_clean(text):
    """Simple cleaning first."""
    # normalize unicode (important!)
    text = unicodedata.normalize("NFKC", text)

    # collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# 3️⃣ Normalize line breaks

PDFs often have random newlines inserted.

In [4]:
def fix_newlines(text):
    # Replace multiple newline with one
    text = re.sub(r'\n\s*\n+', '\n', text)
    # remove isolated single newlines between words
    text = re.sub(r'(?<=\w)\n(?=\w)', ' ', text)
    return text

# 4️⃣ Remove bullet symbols (even if rare)

We detect and remove multiple bullet patterns:

In [5]:
BULLETS = ["•", "●", "▪", "■", "-", "–", "—", "→"]

def remove_bullets(text):
    for b in BULLETS:
        text = text.replace(b, " ")
    return text

# 5️⃣ Remove copyright, trademark noise

Scan first—remove realistic ones:

In [6]:
def remove_legal_symbols(text):
    text = text.replace("©", " ")
    text = text.replace("®", " ")
    return text

# 6️⃣ Put everything together

In [7]:
def clean_text(text):
    text = basic_clean(text)
    text = fix_newlines(text)
    text = remove_bullets(text)
    text = remove_legal_symbols(text)
    text = text.lower()  # final standardization
    return text.strip()

# 7️⃣ Test cleaning function

In [8]:
for name, raw in docs.items():
    print("="*60)
    print(name.upper())
    print(clean_text(raw)[:800])

RESUME_001.PDF
allison hill   environmental health practitioner phone: +91 8410529190 email: jillrhodes@example.net summary: instead ahead despite measure ago curret practice nation. projects: * optimized model using leadership, reducing memory usage by 15%. h skills: project management, angular, attention to detail, leadership, node.js, react experience: english as a second language teache r at lawrence pacheco (2019   2022) fi designed using project management to improve accuracy.   enhanced using leadership to improve memory usage. accounting technician at wolfe llc (2014   2016) n integrated using angular to improve accuracy.   integrated using leadership to improev throughput. fi debbugged using react to improve processing speed. health physicist at lewis llc (2019   2024)   optimized using angular 
RESUME_002.PDF
james howard   local government officer email: lindsay78@example.org | phone: +1 (202) 599 5345 summary: need stop peace technology. court attorney product significant w

# 8️⃣ Save cleaned versions

This helps compare raw vs cleaned.

In [11]:
SAVE_DIR = Path("../../dataset/processed/")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Create subfolders
SAVE_RESUME_DIR = SAVE_DIR / "resumes"
SAVE_JD_DIR = SAVE_DIR / "jds"

SAVE_RESUME_DIR.mkdir(parents=True, exist_ok=True)
SAVE_JD_DIR.mkdir(parents=True, exist_ok=True)

# Save resumes
for pdf in DATASET_RESUME_DIR.glob("*.pdf"):
    raw = extract_pdf_text(pdf)
    cleaned = clean_text(raw)
    (SAVE_RESUME_DIR / f"{pdf.stem}_clean.txt").write_text(cleaned, encoding="utf-8")

# Save job descriptions
for pdf in DATASET_JD_DIR.glob("*.pdf"):
    raw = extract_pdf_text(pdf)
    cleaned = clean_text(raw)
    (SAVE_JD_DIR / f"{pdf.stem}_clean.txt").write_text(cleaned, encoding="utf-8")

SAVE_DIR

WindowsPath('../../dataset/processed')

# 🎯 Summary

We built a cleaning pipeline that:

✔ normalizes unicode  
✔ removes bullet symbols  
✔ collapses whitespace  
✔ fixes random newlines  
✔ removes legal symbols  
✔ lowercases the output  

This cleaned text is now ready for:

- skill extraction
- text embedding
- ML training (classification or ranking)

---